In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from utils import introduce_data 

RANDOM_STATE = 36
SEED = 143

In [2]:
df_train = pd.read_csv('data/CLEANED_train.csv')
df_test = pd.read_csv('data/CLEANED_test.csv')
df_holidays = pd.read_csv('data/holidays_events.csv')
df_oil = pd.read_csv('data/oil.csv')
df_stores = pd.read_csv('data/stores.csv')
df_transactions = pd.read_csv('data/test.csv')

/tmp/ipykernel_2171/2547418077.py:1: DtypeWarning: Columns (0: holiday_type, 1: locale, 2: locale_name, 3: description) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('data/CLEANED_train.csv')


In [3]:
datasets = {
    "Train": df_train,
    "Test": df_test,
    "Holidays": df_holidays,
    "Oil": df_oil,
    "Stores": df_stores,
    "Transactions": df_transactions
}
        
# introduce_data(datasets)

In [4]:
df_train

,Unnamed: 0,id,date,store_nbr,family,sales,onpromotion,year,month,day,...,city,state,store_type,cluster,dcoilwtico,holiday_type,locale,locale_name,description,transferred
0,0,0,2013-01-01,1,AUTOMOTIVE,0.000,0,2013,1,1,...,Quito,Pichincha,D,13,NaN,Holiday,National,Ecuador,Primer dia del ano,1
1,1,1,2013-01-01,1,BABY CARE,0.000,0,2013,1,1,...,Quito,Pichincha,D,13,NaN,Holiday,National,Ecuador,Primer dia del ano,1
2,2,2,2013-01-01,1,BEAUTY,0.000,0,2013,1,1,...,Quito,Pichincha,D,13,NaN,Holiday,National,Ecuador,Primer dia del ano,1
3,3,3,2013-01-01,1,BEVERAGES,0.000,0,2013,1,1,...,Quito,Pichincha,D,13,NaN,Holiday,National,Ecuador,Primer dia del ano,1
4,4,4,2013-01-01,1,BOOKS,0.000,0,2013,1,1,...,Quito,Pichincha,D,13,NaN,Holiday,National,Ecuador,Primer dia del ano,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3054343,3054343,3000883,2017-08-15,9,POULTRY,438.133,0,2017,8,15,...,Quito,Pichincha,B,6,47.57,Holiday,Local,Riobamba,Fundacion de Riobamba,1
3054344,3054344,3000884,2017-08-15,9,PREPARED FOODS,154.553,1,2017,8,15,...,Quito,Pichincha,B,6,47.57,Holiday,Local,Riobamba,Fundacion de Riobamba,1
3054345,3054345,3000885,2017-08-15,9,PRODUCE,2419.729,148,2017,8,15,...,Quito,Pichincha,B,6,47.57,Holiday,Local,Riobamba,Fundacion de Riobamba,1
3054346,3054346,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8,2017,8,15,...,Quito,Pichincha,B,6,47.57,Holiday,Local,Riobamba,Fundacion de Riobamba,1


In [5]:
# ============================================================
# 🔍 PREPROCESSING SAFETY CHECK
# Run this BEFORE preprocess(datasets)
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("🔍 PREPROCESSING SAFETY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# 1. DATASET SHAPES
# ------------------------------------------------------------

print("\n📐 DATASET SHAPES")
print("-" * 70)

for name, df in datasets.items():
    print(f"{name:15} → {df.shape[0]:>10,} rows × {df.shape[1]:>3} columns")


# ------------------------------------------------------------
# 2. MEMORY USAGE
# ------------------------------------------------------------

print("\n💾 MEMORY USAGE")
print("-" * 70)

total_memory = 0

for name, df in datasets.items():
    memory = df.memory_usage(deep=True).sum()
    total_memory += memory

    print(
        f"{name:15} → "
        f"{memory / 1024**2:>10.2f} MB"
    )

print("-" * 70)
print(f"{'TOTAL':15} → {total_memory / 1024**2:>10.2f} MB")


# ------------------------------------------------------------
# 3. BASIC DATAFRAME INFORMATION
# ------------------------------------------------------------

print("\n🧱 DATA TYPES")
print("-" * 70)

for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.dtypes.value_counts())


# ------------------------------------------------------------
# 4. CHECK REQUIRED MERGE COLUMNS
# ------------------------------------------------------------

print("\n🔑 MERGE KEY CHECK")
print("-" * 70)

merge_requirements = {
    "Train": ["store_nbr", "date"],
    "Test": ["store_nbr", "date"],
    "Stores": ["store_nbr"],
    "Oil": ["date"],
    "Holidays": ["date"],
}

for dataset_name, columns in merge_requirements.items():

    df = datasets[dataset_name]

    for column in columns:

        if column in df.columns:
            print(
                f"✅ {dataset_name:10} has '{column}' "
                f"({df[column].dtype})"
            )
        else:
            print(
                f"❌ {dataset_name:10} MISSING '{column}'"
            )


# ------------------------------------------------------------
# 5. DUPLICATE MERGE KEYS
# ------------------------------------------------------------

print("\n🔁 DUPLICATE MERGE KEYS")
print("-" * 70)

# Stores should have exactly one row per store
if "store_nbr" in df_stores.columns:

    duplicate_stores = df_stores["store_nbr"].duplicated().sum()

    print(
        f"Stores / store_nbr duplicates: "
        f"{duplicate_stores:,}"
    )

    if duplicate_stores == 0:
        print("✅ Stores is safe for many-to-one merge")
    else:
        print("🚨 WARNING: Stores contains duplicate store_nbr values!")


# Oil should ideally have one row per date
if "date" in df_oil.columns:

    oil_duplicate_dates = df_oil["date"].duplicated().sum()

    print(
        f"Oil / date duplicates: "
        f"{oil_duplicate_dates:,}"
    )

    if oil_duplicate_dates == 0:
        print("✅ Oil is safe for many-to-one merge")
    else:
        print("🚨 WARNING: Oil contains duplicate dates!")


# Holidays CAN legitimately contain duplicate dates
if "date" in df_holidays.columns:

    holiday_duplicate_dates = df_holidays["date"].duplicated().sum()

    print(
        f"Holidays / date duplicates: "
        f"{holiday_duplicate_dates:,}"
    )

    if holiday_duplicate_dates == 0:
        print("ℹ️ No duplicate holiday dates")
    else:
        print(
            "⚠️ Holidays contains duplicate dates."
        )
        print(
            "   This MAY multiply rows during merge!"
        )


# ------------------------------------------------------------
# 6. DATE DTYPE CHECK
# ------------------------------------------------------------

print("\n📅 DATE TYPE CHECK")
print("-" * 70)

for name in ["Train", "Test", "Oil", "Holidays"]:

    df = datasets[name]

    if "date" in df.columns:

        print(
            f"{name:10} → "
            f"{df['date'].dtype}"
        )


# ------------------------------------------------------------
# 7. MISSING VALUES
# ------------------------------------------------------------

print("\n🕳️ MISSING VALUES")
print("-" * 70)

for name, df in datasets.items():

    missing = df.isna().sum()

    missing = missing[missing > 0]

    print(f"\n{name}:")

    if len(missing) == 0:
        print("  ✅ No missing values")
    else:
        print(
            missing.sort_values(ascending=False).head(15)
        )


# ------------------------------------------------------------
# 8. SIMULATE MERGES
# ------------------------------------------------------------

print("\n🧪 SIMULATING MERGES")
print("-" * 70)

train_test = df_train.copy()

print(
    f"START TRAIN      → "
    f"{len(train_test):,} rows"
)


# ---- Stores ----

before = len(train_test)

train_test = train_test.merge(
    df_stores,
    on="store_nbr",
    how="left"
)

after = len(train_test)

print(
    f"After Stores     → "
    f"{after:,} rows "
    f"({after / before:.2f}x)"
)

if after > before:
    print("🚨 ROW COUNT INCREASED!")



# ---- Oil ----

before = len(train_test)

train_test = train_test.merge(
    df_oil,
    on="date",
    how="left"
)

after = len(train_test)

print(
    f"After Oil        → "
    f"{after:,} rows "
    f"({after / before:.2f}x)"
)

if after > before:
    print("🚨 ROW COUNT INCREASED!")



# ---- Holidays ----

before = len(train_test)

train_test = train_test.merge(
    df_holidays,
    on="date",
    how="left"
)

after = len(train_test)

print(
    f"After Holidays   → "
    f"{after:,} rows "
    f"({after / before:.2f}x)"
)

if after > before:
    print(
        "🚨🚨🚨 HOLIDAY MERGE INCREASED ROW COUNT!"
    )


# ------------------------------------------------------------
# 9. MEMORY AFTER SIMULATED MERGES
# ------------------------------------------------------------

print("\n💾 MEMORY AFTER SIMULATED MERGES")
print("-" * 70)

merged_memory = (
    train_test.memory_usage(deep=True).sum()
)

print(
    f"Original train memory → "
    f"{df_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print(
    f"Merged train memory   → "
    f"{merged_memory / 1024**2:.2f} MB"
)

print(
    f"Memory multiplier     → "
    f"{merged_memory / df_train.memory_usage(deep=True).sum():.2f}x"
)


# ------------------------------------------------------------
# 10. CHECK CATEGORICAL COLUMNS
# ------------------------------------------------------------

print("\n🔤 CATEGORICAL COLUMNS")
print("-" * 70)

categorical_cols = df_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print(f"Found {len(categorical_cols)} categorical columns:")

for column in categorical_cols:

    unique = df_train[column].nunique(dropna=False)

    print(
        f"  {column:20} → "
        f"{unique:,} unique values"
    )


# ------------------------------------------------------------
# 11. ESTIMATE ONE-HOT ENCODING SIZE
# ------------------------------------------------------------

print("\n🧮 ONE-HOT ENCODING ESTIMATE")
print("-" * 70)

total_categories = 0

for column in categorical_cols:

    unique = df_train[column].nunique(dropna=False)
    total_categories += unique

    print(
        f"{column:20} → "
        f"{unique:,} categories"
    )

print("-" * 70)

print(
    f"Total categorical features: "
    f"{total_categories:,}"
)

print(
    f"Approximate sparse matrix dimensions: "
    f"{len(df_train):,} × "
    f"{total_categories:,}"
)


# ------------------------------------------------------------
# 12. CHECK EXTREME CARDINALITY
# ------------------------------------------------------------

print("\n🚨 HIGH-CARDINALITY COLUMNS")
print("-" * 70)

for column in categorical_cols:

    unique = df_train[column].nunique(dropna=False)

    if unique > 100:
        print(
            f"⚠️ {column:20} → "
            f"{unique:,} unique categories"
        )


# ------------------------------------------------------------
# 13. CLEAN UP SIMULATION COPY
# ------------------------------------------------------------

del train_test


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ SAFETY CHECK COMPLETE")
print("=" * 70)

print(
    "\nIf you see 🚨 warnings above, "
    "DO NOT run the full preprocess() yet."
)

print(
    "\nMost important things to look at:"
)

print(
    "  1. Row count after each merge"
)

print(
    "  2. Memory multiplier after merges"
)

print(
    "  3. Duplicate dates in Holidays"
)

print(
    "  4. Number of categorical features"
)

print(
    "  5. Total dataset memory"
)

print("=" * 70)

🔍 PREPROCESSING SAFETY CHECK

📐 DATASET SHAPES
----------------------------------------------------------------------
Train           →  3,054,348 rows ×  22 columns
Test            →     28,512 rows ×  21 columns
Holidays        →        350 rows ×   6 columns
Oil             →      1,218 rows ×   2 columns
Stores          →         54 rows ×   5 columns
Transactions    →     28,512 rows ×   5 columns

💾 MEMORY USAGE
----------------------------------------------------------------------
Train           →    1551.85 MB
Test            →      14.61 MB
Holidays        →       0.10 MB
Oil             →       0.08 MB
Stores          →       0.01 MB
Transactions    →       3.88 MB
----------------------------------------------------------------------
TOTAL           →    1570.53 MB

🧱 DATA TYPES
----------------------------------------------------------------------

Train:
int64      11
str         9
float64     2
Name: count, dtype: int64

Test:
int64      10
str         9
float64     1
ob

In [6]:
1/0

ZeroDivisionError: division by zero

In [ ]:
def preprocess(datasets):
    
    df_train = datasets["Train"]
    df_test = datasets["Test"]
    df_holidays = datasets["Holidays"]
    df_oil = datasets["Oil"]
    df_stores = datasets["Stores"]
    df_transactions = datasets["Transactions"]
            
    # Dropping unused columns
    df_train = df_train.drop(['id', 'date'], axis=1)
    df_test = df_test.drop(['id', 'date'], axis=1)
    
    # Seperating df_train, y_train
    y_train = df_train['sales']
    df_train = df_train.drop('sales', axis=1)
    
    categorical_cols = df_train.select_dtypes('str').columns
    
    col_transformer = ColumnTransformer([
        ('one_hot_encode', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ], remainder='passthrough')
    
    df_train_encoded = col_transformer.fit_transform(df_train)
    df_test_encoded = col_transformer.transform(df_test)
    
    return df_train_encoded, y_train, df_test_encoded

df_train, y_train, df_test = preprocess(datasets)

In [ ]:
print(df_train.shape, df_test.shape)
# print(df_train.columns, df_test.columns) 
df_train.head()

In [ ]:
# df_train.to_csv('data/CLEANED_train.csv')
# df_test.to_csv('data/CLEANED_test.csv')